# RSNA Knee Abnormality Detection - Baseline_v2 Inference

This notebook is aligned with the baseline_v2 training configuration:

- DINOv2 HuggingFace backbone
- image size: 336
- slices per series: 32
- physical crop: 140 mm
- inference automatically adapts between 1 and 2 GPUs

The preprocessing parameters should match the training pipeline.


In [ ]:
"""RSNA Knee 数据加载与 140 mm physical crop 预处理模块。"""

from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


KAGGLE_INPUT = Path("/kaggle/input")


def find_data_root(root=None):
    """Return the mounted Kaggle dataset containing the RSNA CSV files.

    Pass ``--data-root`` to train.py if more than one mounted dataset contains
    a train.csv.  This keeps the notebook independent of Kaggle dataset slugs.
    """
    if root is not None:
        root = Path(root)
        if (root / "train.csv").is_file():
            return root
        raise FileNotFoundError(f"RSNA data root has no train.csv: {root}")
    candidates = [path.parent for path in KAGGLE_INPUT.rglob("train.csv")
                  if (path.parent / "train_series.csv").is_file()
                  and (path.parent / "test.csv").is_file()
                  and (path.parent / "test_series.csv").is_file()]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "Could not find RSNA data under /kaggle/input. Attach the competition "
            "data, or provide --data-root /kaggle/input/<your-data-dataset>."
        )
    raise RuntimeError(
        "Multiple possible RSNA data roots found: " + ", ".join(map(str, candidates)) +
        ". Provide --data-root explicitly."
    )


DEFAULT_ROOT = None
DEFAULT_IMAGE_SIZE = 336
DEFAULT_NUM_SLICES = 32
DEFAULT_CROP_MM = 140.0
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = ["Axial_FS", "Axial_nonFS", "Coronal_FS", "Coronal_nonFS", "Sagittal_FS", "Sagittal_nonFS"]
SERIES_FEATURES = ["fluid_sensitive", "fat_suppression", "t1", "pd", "t2", "contrast_unknown", "te", "tr", "pixel_spacing", "slice_spacing", "coverage"]


def _count_dicoms(path):
    return sum(1 for _ in Path(path).glob("*.dcm"))


def _prepare_series_df(series_df, image_root):
    series_df = series_df.copy()
    series_df["SeriesPath"] = [Path(image_root) / str(study) / str(series) for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID)]
    series_df["SeriesSlot"] = series_df.Anatomical_Plane + "_" + np.where(series_df.Fat_Suppression.eq(1), "FS", "nonFS")
    series_df["NumSlices"] = [_count_dicoms(path) for path in series_df.SeriesPath]
    return series_df


def load_metadata(root=DEFAULT_ROOT):
    root = find_data_root(root)
    train_df, test_df = pd.read_csv(root / "train.csv"), pd.read_csv(root / "test.csv")
    train_series_df = _prepare_series_df(pd.read_csv(root / "train_series.csv"), root / "train_series")
    test_series_df = _prepare_series_df(pd.read_csv(root / "test_series.csv"), root / "test_series")
    return train_df, train_series_df, test_df, test_series_df, pd.read_csv(root / "sample_submission.csv")


def get_sorted_dicom_info(series_path):
    """使用患者坐标排序，并把主切片方向统一为患者坐标轴的正方向。"""
    files = list(Path(series_path).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"No DICOM files found in series: {series_path}")
    headers = [pydicom.dcmread(file, stop_before_pixels=True) for file in files]
    orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=np.float32)
    normal = np.cross(orientation[:3], orientation[3:])
    normal /= np.linalg.norm(normal)
    if normal[np.argmax(np.abs(normal))] < 0:
        normal = -normal
    positions = np.asarray([np.dot(np.asarray(header.ImagePositionPatient, dtype=np.float32), normal) for header in headers])
    order = np.argsort(positions)
    return [files[i] for i in order], positions[order], [headers[i] for i in order]


def sort_dicom_files(series_path):
    return get_sorted_dicom_info(series_path)[0]


def read_dicom(path):
    dicom = pydicom.dcmread(path)
    try:
        image = dicom.pixel_array.astype(np.float32)
    except ValueError as error:
        # 少数文件的头信息写着16位，但未压缩PixelData实际是8位；按真实字节数恢复。
        rows, columns = int(dicom.Rows), int(dicom.Columns)
        frames = int(getattr(dicom, "NumberOfFrames", 1) or 1)
        samples = int(getattr(dicom, "SamplesPerPixel", 1) or 1)
        pixel_count = rows * columns * frames * samples
        raw = dicom.PixelData
        transfer_syntax = getattr(getattr(dicom, "file_meta", None), "TransferSyntaxUID", None)
        uncompressed = transfer_syntax is None or not transfer_syntax.is_compressed
        if not uncompressed or frames != 1 or samples != 1 or len(raw) not in (pixel_count, pixel_count + 1):
            raise ValueError(f"Cannot decode DICOM {path}: {error}") from error
        dtype = np.int8 if int(getattr(dicom, "PixelRepresentation", 0) or 0) else np.uint8
        image = np.frombuffer(raw[:pixel_count], dtype=dtype).reshape(rows, columns).astype(np.float32)
    image = image * float(getattr(dicom, "RescaleSlope", 1)) + float(getattr(dicom, "RescaleIntercept", 0))
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() + image.min() - image
    return image


def _read_sampled_slices(files, indices):
    """读取采样切片；单张损坏时用空间上最近的可读切片替代。"""
    cache, invalid = {}, set()
    images = []
    for index in indices:
        index = int(index)
        candidates = sorted(range(len(files)), key=lambda candidate: abs(candidate - index))
        for candidate in candidates:
            if candidate in cache:
                image = cache[candidate]
                break
            if candidate in invalid:
                continue
            try:
                image = read_dicom(files[candidate])
                cache[candidate] = image
                break
            except (ValueError, OSError, EOFError, InvalidDicomError):
                invalid.add(candidate)
        else:
            raise RuntimeError(f"No decodable DICOM slices in series: {Path(files[0]).parent}")
        images.append(image)
    return np.stack(images)


def _sample_by_position(positions, num_slices):
    """在真实物理覆盖范围内均匀采样，而不是只按切片编号采样。"""
    targets = np.linspace(positions[0], positions[-1], num_slices)
    return np.abs(positions[:, None] - targets[None, :]).argmin(axis=0)


def _get_pixel_spacing(headers, series_path):
    """读取有效的行、列 PixelSpacing；优先使用堆栈中央切片。"""
    middle = len(headers) // 2
    search_order = sorted(range(len(headers)), key=lambda index: abs(index - middle))
    for index in search_order:
        value = getattr(headers[index], "PixelSpacing", None)
        if value is None or len(value) < 2:
            continue
        spacing = np.asarray(value[:2], dtype=np.float32)
        if np.all(np.isfinite(spacing)) and np.all(spacing > 0):
            return spacing
    raise ValueError(f"Series has no valid PixelSpacing: {series_path}")


def physical_center_crop(volume, pixel_spacing, crop_mm=DEFAULT_CROP_MM):
    """按真实毫米尺度中心裁剪；原始 FOV 不足时先对称补零以保持尺度。"""
    if volume.ndim != 3:
        raise ValueError(f"Expected volume [S,H,W], got shape {volume.shape}")
    if not np.isfinite(crop_mm) or crop_mm <= 0:
        raise ValueError(f"crop_mm must be positive, got {crop_mm}")

    spacing = np.asarray(pixel_spacing, dtype=np.float32)
    if spacing.shape != (2,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
        raise ValueError(f"Invalid PixelSpacing: {pixel_spacing}")

    target_height = max(1, int(round(crop_mm / float(spacing[0]))))
    target_width = max(1, int(round(crop_mm / float(spacing[1]))))
    height, width = volume.shape[-2:]

    pad_height = max(0, target_height - height)
    pad_width = max(0, target_width - width)
    if pad_height or pad_width:
        volume = np.pad(
            volume,
            (
                (0, 0),
                (pad_height // 2, pad_height - pad_height // 2),
                (pad_width // 2, pad_width - pad_width // 2),
            ),
            mode="constant",
            constant_values=0,
        )
        height, width = volume.shape[-2:]

    top = (height - target_height) // 2
    left = (width - target_width) // 2
    return volume[:, top:top + target_height, left:left + target_width]


def _contrast_features(te, tr, fat_suppression):
    """依据 EDA 使用的 TE/TR 启发式生成 T1、PD、T2、未知四维标记。"""
    if fat_suppression:
        return [0, 1, 1, 0]
    if te <= 0 or tr <= 0:
        return [0, 0, 0, 1]
    if tr < 1000 and te < 30:
        return [1, 0, 0, 0]
    if te < 60:
        return [0, 1, 0, 0]
    return [0, 0, 1, 0]


def load_series(
    series_path,
    num_slices=DEFAULT_NUM_SLICES,
    image_size=DEFAULT_IMAGE_SIZE,
    crop_mm=DEFAULT_CROP_MM,
    fluid_sensitive=0,
    fat_suppression=0,
    return_features=False,
):
    """空间排序→物理采样→解码→140 mm crop→归一化→336 px resize。"""
    files, positions, headers = get_sorted_dicom_info(series_path)
    indices = _sample_by_position(positions, num_slices)
    volume = _read_sampled_slices(files, indices)
    spacing = _get_pixel_spacing(headers, series_path)
    volume = physical_center_crop(volume, spacing, crop_mm)

    foreground = volume[np.isfinite(volume) & (volume != 0)]
    if not foreground.size:
        raise ValueError(f"Series contains no finite non-zero pixels after crop: {series_path}")
    low, high = np.percentile(foreground, [0.5, 99.5])
    volume = np.clip((volume - low) / (high - low + 1e-6), 0, 1).astype(np.float32)
    volume = torch.from_numpy(volume)[:, None]
    volume = F.interpolate(volume, size=(image_size, image_size), mode="bilinear", align_corners=False)

    header = headers[len(headers) // 2]
    te, tr = float(getattr(header, "EchoTime", 0) or 0), float(getattr(header, "RepetitionTime", 0) or 0)
    slice_spacing = float(np.median(np.abs(np.diff(positions)))) if len(positions) > 1 else 0
    coverage = float(abs(positions[-1] - positions[0])) if len(positions) > 1 else 0
    contrast = _contrast_features(te, tr, int(fat_suppression))
    features = torch.tensor([float(fluid_sensitive), float(fat_suppression), *contrast, te / 100, tr / 5000, float(spacing.mean()), slice_spacing / 10, coverage / 200], dtype=torch.float32)
    return (volume, features) if return_features else volume


class KneeDataset(Dataset):
    """Study 级数据集，输出 images [6,S,1,H,W] 和六个槽位的元数据。"""

    def __init__(
        self,
        study_df,
        series_df,
        num_slices=DEFAULT_NUM_SLICES,
        image_size=DEFAULT_IMAGE_SIZE,
        crop_mm=DEFAULT_CROP_MM,
    ):
        self.study_df = study_df.reset_index(drop=True)
        self.num_slices, self.image_size, self.crop_mm = num_slices, image_size, crop_mm
        self.series_groups = {uid: group.reset_index(drop=True) for uid, group in series_df.groupby("StudyInstanceUID")}

    def __len__(self):
        return len(self.study_df)

    def __getitem__(self, index):
        row, uid = self.study_df.iloc[index], self.study_df.iloc[index].StudyInstanceUID
        series_df = self.series_groups[uid]
        images = torch.zeros(len(SLOTS), self.num_slices, 1, self.image_size, self.image_size)
        slot_mask = torch.zeros(len(SLOTS))
        series_features = torch.zeros(len(SLOTS), len(SERIES_FEATURES))

        for slot_index, slot in enumerate(SLOTS):
            candidates = series_df[series_df.SeriesSlot == slot]
            if len(candidates):
                selected = candidates.sort_values(["Fluid_Sensitive", "NumSlices"], ascending=[False, False]).iloc[0]
                volume, features = load_series(
                    selected.SeriesPath,
                    self.num_slices,
                    self.image_size,
                    self.crop_mm,
                    selected.Fluid_Sensitive,
                    selected.Fat_Suppression,
                    True,
                )
                images[slot_index], series_features[slot_index], slot_mask[slot_index] = volume, features, 1

        values = pd.to_numeric(row[LABELS], errors="coerce").to_numpy(dtype=np.float32) if set(LABELS).issubset(row.index) else np.full(len(LABELS), np.nan, dtype=np.float32)
        valid = ~np.isnan(values)
        label_weight = np.where(valid, 2 * np.abs(values - 0.5), 0).astype(np.float32)
        return {
            "study_uid": uid,
            "images": images,
            "slot_mask": slot_mask,
            "series_features": series_features,
            "targets": torch.from_numpy(np.nan_to_num(values, nan=0)),
            "label_mask": torch.from_numpy(valid.astype(np.float32)),
            "label_weight": torch.from_numpy(label_weight),
        }

def build_datasets(root=DEFAULT_ROOT, num_slices=DEFAULT_NUM_SLICES, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM):
    metadata = load_metadata(root)
    train_df, train_series_df, test_df, test_series_df, sample_submission = metadata
    train_dataset = KneeDataset(train_df, train_series_df, num_slices, image_size, crop_mm)
    test_dataset = KneeDataset(test_df, test_series_df, num_slices, image_size, crop_mm)
    return train_dataset, test_dataset, sample_submission, metadata


def build_dataloaders(
    root=DEFAULT_ROOT,
    batch_size=2,
    num_workers=4,
    num_slices=DEFAULT_NUM_SLICES,
    image_size=DEFAULT_IMAGE_SIZE,
    crop_mm=DEFAULT_CROP_MM,
):
    train_dataset, test_dataset, sample_submission, metadata = build_datasets(root, num_slices, image_size, crop_mm)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader, sample_submission, metadata



__all__ = ["DEFAULT_ROOT", "DEFAULT_IMAGE_SIZE", "DEFAULT_NUM_SLICES", "DEFAULT_CROP_MM", "LABELS", "SLOTS", "SERIES_FEATURES", "find_data_root", "load_metadata", "get_sorted_dicom_info", "sort_dicom_files", "read_dicom", "physical_center_crop", "load_series", "KneeDataset", "build_datasets", "build_dataloaders"]


In [ ]:
"""DINOv2-based hierarchical model for RSNA Knee."""

from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModel



def load_backbone(model_dir):
    """Load the Kaggle-attached Hugging Face DINOv2 model without network access."""
    model_dir = Path(model_dir)
    required = [model_dir / "config.json", model_dir / "pytorch_model.bin"]
    missing = [str(path.name) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"DINOv2 Hugging Face model directory is incomplete: {model_dir}; missing {missing}"
        )
    return AutoModel.from_pretrained(str(model_dir), local_files_only=True)


class RSNADINOv2(nn.Module):
    """2.5D DINOv2 encoder + slice aggregation + label-specific slot fusion."""

    def __init__(
        self,
        model_dir=None,
        hidden_dim=256,
        num_slices=24,
        num_heads=8,
        slice_layers=2,
        slot_layers=1,
        dropout=0.1,
        freeze_backbone=True,
        encoder_chunk_size=24,
    ):
        super().__init__()
        self.backbone = load_backbone(model_dir)

        self.num_slices = num_slices
        self.encoder_chunk_size = encoder_chunk_size
        self.freeze_backbone(freeze_backbone)
        backbone_dim = self.backbone.config.hidden_size

        self.register_buffer("image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.slice_projection = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, hidden_dim))
        self.slice_position = nn.Parameter(torch.zeros(1, num_slices, hidden_dim))

        slice_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slice_encoder = nn.TransformerEncoder(slice_layer, slice_layers, enable_nested_tensor=False)
        self.slice_attention = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 1))

        self.metadata_projection = nn.Sequential(
            nn.Linear(len(SERIES_FEATURES), hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim),
        )
        self.slot_embedding = nn.Parameter(torch.zeros(1, len(SLOTS), hidden_dim))
        slot_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slot_encoder = nn.TransformerEncoder(slot_layer, slot_layers, enable_nested_tensor=False)

        self.label_queries = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.label_weight = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_bias = nn.Parameter(torch.zeros(len(LABELS)))
        nn.init.trunc_normal_(self.slice_position, std=0.02)
        nn.init.trunc_normal_(self.slot_embedding, std=0.02)
        nn.init.trunc_normal_(self.label_queries, std=0.02)
        nn.init.trunc_normal_(self.label_weight, std=0.02)

    def freeze_backbone(self, freeze=True):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze

    def unfreeze_last_blocks(self, num_blocks=2):
        self.freeze_backbone(True)
        for block in self.backbone.encoder.layer[-num_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in self.backbone.layernorm.parameters():
            parameter.requires_grad = True

    @staticmethod
    def make_2p5d(images):
        """把前一张、当前张、后一张切片组成 DINOv2 的三个输入通道。"""
        gray = images.squeeze(3)
        previous = torch.cat([gray[:, :, :1], gray[:, :, :-1]], dim=2)
        following = torch.cat([gray[:, :, 1:], gray[:, :, -1:]], dim=2)
        return torch.stack([previous, gray, following], dim=3)

    def encode_images(self, images, slot_mask):
        batch, slots, slices, _, height, width = images.shape
        images = self.make_2p5d(images).reshape(batch * slots * slices, 3, height, width)
        valid = slot_mask[:, :, None].expand(batch, slots, slices).reshape(-1).bool()
        valid_indices = valid.nonzero(as_tuple=False).squeeze(1)
        images = images[valid_indices]
        images = (images - self.image_mean) / self.image_std

        use_grad = any(parameter.requires_grad for parameter in self.backbone.parameters())
        context = nullcontext() if use_grad else torch.no_grad()
        with context:
            chunks = [self.backbone(pixel_values=chunk).last_hidden_state[:, 0] for chunk in images.split(self.encoder_chunk_size)]
        encoded = torch.cat(chunks)
        all_encoded = encoded.new_zeros(batch * slots * slices, encoded.shape[-1])
        all_encoded = all_encoded.index_copy(0, valid_indices, encoded)
        return all_encoded.view(batch, slots, slices, -1)

    def forward(self, images, slot_mask, series_features=None):
        batch, slots, slices = images.shape[:3]
        if slices != self.num_slices:
            raise ValueError(f"Expected {self.num_slices} slices, got {slices}")

        features = self.encode_images(images, slot_mask)
        features = self.slice_projection(features).view(batch * slots, slices, -1)
        features = self.slice_encoder(features + self.slice_position)
        attention = self.slice_attention(features).softmax(dim=1)
        slot_features = (features * attention).sum(dim=1).view(batch, slots, -1)

        if series_features is not None:
            slot_features = slot_features + self.metadata_projection(series_features)
        slot_features = self.slot_encoder(slot_features + self.slot_embedding, src_key_padding_mask=~slot_mask.bool())

        queries = self.label_queries.unsqueeze(0).expand(batch, -1, -1)
        label_features, _ = self.label_attention(queries, slot_features, slot_features, key_padding_mask=~slot_mask.bool())
        label_features = self.output_norm(label_features)
        return (label_features * self.label_weight.unsqueeze(0)).sum(dim=-1) + self.label_bias


__all__ = ["RSNADINOv2"]


In [ ]:
"""Kaggle submission inference for checkpoints trained by this package."""

import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader



# Kaggle input paths confirmed by the successful run; edit here if mounts change.
DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CHECKPOINT_PATH = "/kaggle/input/datasets/yzzzh7/baseline-v1-weights/best.pt"
DINOV2_MODEL_DIR = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")
BATCH_SIZE = 1
NUM_WORKERS = 0
ENCODER_CHUNK_SIZE = 16
# Use up to two Kaggle GPUs; a single GPU works too.
NUM_GPUS = min(2, torch.cuda.device_count())


def find_one(candidates, description):
    paths = sorted(set(Path(path) for path in candidates))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {description}, found {len(paths)}: {paths}. Set its path explicitly above.")
    return paths[0]


def input_files():
    """Find small input metadata without descending into millions of DICOM files."""
    for directory, subdirs, files in os.walk("/kaggle/input"):
        subdirs[:] = [name for name in subdirs if name not in {"train_series", "test_series"}]
        here = Path(directory)
        for name in files:
            if name in {"test_series.csv", "best.pt", "config.json"}:
                yield here / name


def resolve_inputs():
    found = list(input_files()) if not all((DATA_ROOT, CHECKPOINT_PATH, DINOV2_MODEL_DIR)) else []
    root = Path(DATA_ROOT) if DATA_ROOT else find_one(
        (p.parent for p in found if p.name == "test_series.csv"
         if (p.parent / "test.csv").is_file()
         and (p.parent / "sample_submission.csv").is_file()
         and (p.parent / "test_series").is_dir()),
        "competition data directory",
    )
    checkpoint = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else find_one(
        (p for p in found if p.name == "best.pt"), "best.pt checkpoint"
    )
    model_dir = Path(DINOV2_MODEL_DIR) if DINOV2_MODEL_DIR else find_one(
        (p.parent for p in found if p.name == "config.json"
         if (p.parent / "pytorch_model.bin").is_file()),
        "Hugging Face DINOv2 directory",
    )
    for path in (root / "test.csv", root / "test_series.csv", root / "sample_submission.csv",
                 root / "test_series", checkpoint, model_dir / "config.json",
                 model_dir / "pytorch_model.bin"):
        if not path.exists():
            raise FileNotFoundError(path)
    return root, checkpoint, model_dir


@torch.inference_mode()
def inference_worker(rank, model, local_test_df, series_df, num_slices, image_size, crop_mm, root):
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")
    started = time.monotonic()
    series_df = series_df[series_df.StudyInstanceUID.isin(local_test_df.StudyInstanceUID)]
    print(f"GPU {rank}: indexing {len(series_df)} local series", flush=True)
    series_df = _prepare_series_df(series_df, root / "test_series")
    missing = set(local_test_df.StudyInstanceUID) - set(series_df.StudyInstanceUID)
    if missing:
        raise ValueError(f"No series metadata for {len(missing)} test studies")
    dataset = KneeDataset(local_test_df, series_df, num_slices=num_slices, image_size=image_size, crop_mm=crop_mm)
    print(f"GPU {rank}: {len(dataset)} studies assigned", flush=True)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0)

    predictions = {}
    for index, batch in enumerate(loader, start=1):
        if index == 1:
            print(f"GPU {rank}: first study loaded after {time.monotonic() - started:.1f}s", flush=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = model(batch["images"].to(device, non_blocking=True),
                           batch["slot_mask"].to(device, non_blocking=True),
                           batch["series_features"].to(device, non_blocking=True))
        scores = torch.sigmoid(logits.float()).cpu().numpy()
        for uid, score in zip(batch["study_uid"], scores):
            predictions[str(uid)] = score
        if index % 5 == 0 or index == len(loader):
            print(f"GPU {rank}: {min(index * BATCH_SIZE, len(dataset))}/{len(dataset)} studies, "
                  f"elapsed {time.monotonic() - started:.1f}s", flush=True)

    if len(predictions) != len(dataset):
        raise ValueError(f"Expected {len(dataset)} predictions, got {len(predictions)}")
    values = np.stack([predictions[uid] for uid in local_test_df.StudyInstanceUID])
    if values.shape != (len(dataset), len(LABELS)) or not np.isfinite(values).all():
        raise ValueError("Prediction shape or finiteness check failed")
    if not ((values >= 0) & (values <= 1)).all():
        raise ValueError("Predictions are outside [0, 1]")
    partial = pd.DataFrame(values, columns=LABELS)
    partial.insert(0, "StudyInstanceUID", local_test_df.StudyInstanceUID.to_numpy())
    return partial


def main():
    root, checkpoint_path, model_dir = resolve_inputs()
    print(f"data={root}\ncheckpoint={checkpoint_path}\nDINOv2={model_dir}", flush=True)
    if NUM_GPUS < 1 or NUM_GPUS > torch.cuda.device_count():
        raise RuntimeError(f"Enable a Kaggle GPU accelerator; found {torch.cuda.device_count()} GPU(s).")
    test_df = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    series_df = pd.read_csv(root / "test_series.csv", dtype={
        "StudyInstanceUID": str, "SeriesInstanceUID": str,
    })
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    expected = ["StudyInstanceUID", *LABELS]
    if list(sample.columns) != expected:
        raise ValueError(f"Submission columns differ from the training model: {list(sample.columns)}")
    if test_df.StudyInstanceUID.duplicated().any() or sample.StudyInstanceUID.duplicated().any():
        raise ValueError("Duplicate StudyInstanceUID in test or sample_submission")
    if set(test_df.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("test.csv and sample_submission.csv contain different studies")

    print("Loading checkpoint", flush=True)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    state = checkpoint["model"] if "model" in checkpoint else checkpoint
    if any(key.startswith("backbone.patch_embed.") for key in state):
        raise ValueError("This best.pt uses the old torch.hub DINOv2 backbone, not the Hugging Face model")
    args = checkpoint.get("args")
    if not isinstance(args, dict):
        raise ValueError("Checkpoint must contain training args with num_slices, image_size, and crop_mm")
    required_args = ("num_slices", "image_size", "crop_mm")
    missing_args = [name for name in required_args if name not in args]
    if missing_args:
        raise ValueError(f"Checkpoint is missing preprocessing parameters: {missing_args}")
    num_slices = int(args["num_slices"])
    image_size = int(args["image_size"])
    crop_mm = float(args["crop_mm"])
    if num_slices <= 0 or image_size <= 0 or not np.isfinite(crop_mm) or crop_mm <= 0:
        raise ValueError(f"Invalid preprocessing parameters: slices={num_slices}, image_size={image_size}, crop_mm={crop_mm}")
    if int(state["slice_position"].shape[1]) != num_slices:
        raise ValueError("Checkpoint args and slice_position disagree about num_slices")
    print(f"Preprocessing: slices={num_slices}, image_size={image_size}, crop_mm={crop_mm:g}", flush=True)
    models = []
    for rank in range(NUM_GPUS):
        model = RSNADINOv2(model_dir=str(model_dir), num_slices=num_slices,
                           encoder_chunk_size=ENCODER_CHUNK_SIZE, freeze_backbone=True)
        model.load_state_dict(state, strict=True)
        model.to(torch.device(f"cuda:{rank}")).eval()
        models.append(model)
        print(f"GPU {rank}: model ready", flush=True)
    del checkpoint, state

    # A thread per GPU keeps all computation inside this notebook process.
    # The models and studies are disjoint; no gradient or cross-GPU sync is needed.
    with ThreadPoolExecutor(max_workers=NUM_GPUS) as pool:
        futures = [pool.submit(inference_worker, rank, models[rank],
                               test_df.iloc[rank::NUM_GPUS].reset_index(drop=True),
                               series_df, num_slices, image_size, crop_mm, root)
                   for rank in range(NUM_GPUS)]
        partials = [future.result() for future in futures]
    predictions = pd.concat(partials, ignore_index=True)
    if predictions.StudyInstanceUID.duplicated().any():
        raise ValueError("Duplicate predictions across GPUs")
    if set(predictions.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("GPU predictions do not cover the complete sample submission")
    submission = sample[["StudyInstanceUID"]].merge(predictions, on="StudyInstanceUID",
                                                       how="left", validate="one_to_one")
    if list(submission.columns) != ["StudyInstanceUID", *LABELS]:
        raise ValueError("Incorrect submission column order")
    if not np.isfinite(submission[LABELS].to_numpy()).all():
        raise ValueError("Non-finite value in merged submission")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {OUTPUT_PATH}: {submission.shape[0]} rows, {submission.shape[1]} columns")
    print(submission.head(3).to_string(index=False))


if __name__ == "__main__":
    main()
